In [ ]:
import pandas as pd
import numpy as np
import phonenumbers
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
import random

Data Loading and Initial Inspection

In [ ]:
sales_raw = pd.read_csv('sales_raw.csv')
customers_raw = pd.read_csv('customers_raw.csv')
products_raw = pd.read_csv('products_raw.csv') 

Data cleansing definition

In [ ]:
data_quality_rpt = []

def cleanse_data(df: pd.DataFrame):
    numofcol = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
    catCol = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    for col, cnt in df.isna().sum().items():
        if cnt > 0: 
            missingCol = ''.join(f"{col}:{cnt}") 
            if col in numofcol:
                df[col] = df[col].fillna(df[col].median())
            else:
                df[col] = df[col].fillna('Unknown' + str(random.randint(1, 10)))

    dtCol = df.columns[df.columns.str.contains('date', case='False')]
    if len(dtCol) != 0:
        df[dtCol[0]] = pd.to_datetime(df[dtCol[0]], format='mixed', errors='coerce').dt.strftime("%Y-%m-%d")

    transform_df = df.drop_duplicates(keep='first')

    data_quality_rpt.append({
        'Record Count': len(df),
        'Duplicate Rows': int(df.duplicated().sum()),
        'Null Count': numofcol,
        'Insert Count': len(transform_df) 
    })

    return transform_df, data_quality_rpt

Data cleansing for removal of initial letter from keys, date format, phone number standardization and creating new data matching the database columns

In [ ]:
data = {
    'Record Count',
    'Duplicate_Rows',
    'Null_Count',
    'Inser_Count'
}

sales, sales_df = cleanse_data(sales_raw)
customer, customer_df = cleanse_data(customers_raw)
product, product_df = cleanse_data(products_raw)



customer['customer_id'] = customer['customer_id'].astype(str).str.replace(r'\D+', '', regex=True)
customer['phone'] = customer['phone'].apply(lambda x: phonenumbers.format_number(phonenumbers.parse(x, 'IN'), phonenumbers.PhoneNumberFormat.E164))

sales['transaction_id'] = sales['transaction_id'].astype(str).str.replace(r'\D+',"", regex=True)
sales['customer_id'] = sales['customer_id'].astype(str).str.replace(r'\D+',"", regex=True)
sales['product_id'] = sales['product_id'].astype(str).str.replace(r'\D+',"", regex=True)
sales['total_amount'] = sales['quantity'] * sales['unit_price'] 

sales.replace(['N/A', 'NaT', 'nan', '', ' '], np.nan, inplace=True)
sales = sales.dropna()
product.replace(['N/A', 'NaT', 'nan', '', ' '], np.nan, inplace=True)
product = product.dropna()


product['category'] = product['category'].str.title()
newsales = sales
newsales = newsales.rename(columns={'transaction_id': 'order_id'})
newsales = newsales.rename(columns={'transaction_date': 'order_date'})
newsales['order_item_id'] = newsales['order_id']+newsales['product_id']

#import re 

#count = len(product)

#for i in range(count):
#    print(product['category'][i])
    #product['category'][i] = re.sub(r'_(\w)'), lambda m: m.group(1).upper(), product['category'][i]
customer.head(40)
#newsales.head()

flipkart_bits database connection details and loading the data using insert statements. Use the .env file to grab all the DB connection string parameters 

In [ ]:
import mysql.connector
import os
from dotenv import load_dotenv

load_dotenv()


def dataupdate_db(df: pd.DataFrame, table_name):
    connection_string = mysql.connector.connect(host = os.getenv('MYSQL_HOST_URL'), user = os.getenv('MYSQL_USER'), 
                                                password = os.getenv('MYSQL_PASSWORD'), database = os.getenv('MYSQL_DATABASE'))


#    connection_string = mysql.connector.connect(host = 'localhost', user = 'root', 
 #                                               password = 'a6529644', database = 'ecommerce')

    crsr = connection_string.cursor()

    cols = ", ".join(df.columns)
    plceholder = ",".join(["%s"] * len(df.columns))
    print(cols)
    
    sql = f"insert into {os.getenv('MYSQL_DATABASE')}.{table_name}({cols}) values ({plceholder})"
    print(sql)
    crsr.executemany(sql, df.values.tolist())

    connection_string.commit()

    crsr.close()
    connection_string.close()

Product table's product id making it null by removing P at the beginning

In [ ]:
product['product_id'] = product['product_id'].astype(str).str.replace(r'\D+',"", regex=True)
product.head()

Actual loading of the various data elements in the DB by calling above Definition.
**Do Not Run again as the data is already loaded as part of previous run

In [ ]:
#sales.head()
dataupdate_db(customer[['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'registration_date']], 'customers')
dataupdate_db(newsales[['order_id', 'customer_id', 'order_date', 'total_amount', 'status']], 'orders')
dataupdate_db(product[['product_id', 'product_name', 'category', 'price', 'stock_quantity']], 'products')
dataupdate_db(newsales[['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'subtotal']], 'order_items')

Load the report in text file

In [ ]:
file_path = 'data_quality_report.txt'
with open(file_path, 'w') as file:
    for item in data_quality_rpt:
        file.write(f"{item}\n")